# Statistics & Linear Algebra
A practical overview using `numpy`, `scipy`, and `statistics` (built-in). No deep math — just the concepts you'll actually use.

## 1. Descriptive Statistics

In [ ]:
import numpy as np
import statistics as stats  # Python's built-in stats module

data = [12, 15, 14, 10, 18, 22, 14, 16, 13, 19]

# Central tendency — where is the 'center' of the data?
print('Mean  :', np.mean(data))        # average — sensitive to outliers
print('Median:', np.median(data))      # middle value — robust to outliers
print('Mode  :', stats.mode(data))     # most frequent value

# Spread — how spread out is the data?
print('Std Dev  :', np.std(data))      # average distance from the mean
print('Variance :', np.var(data))      # std dev squared
print('Range    :', np.ptp(data))      # max - min
print('IQR      :', np.percentile(data, 75) - np.percentile(data, 25))  # interquartile range

# Percentiles
print('25th percentile:', np.percentile(data, 25))  # Q1
print('75th percentile:', np.percentile(data, 75))  # Q3
print('90th percentile:', np.percentile(data, 90))

## 2. Probability Distributions

In [ ]:
from scipy import stats as sp_stats
import matplotlib.pyplot as plt

# ── Normal Distribution ───────────────────────────────────────────────────────
# Bell curve — most natural phenomena follow this (heights, test scores, errors)
# Defined by mean (μ) and standard deviation (σ)
mu, sigma = 70, 10
x = np.linspace(30, 110, 200)

pdf = sp_stats.norm.pdf(x, mu, sigma)   # PDF: probability density at each x
cdf = sp_stats.norm.cdf(x, mu, sigma)   # CDF: probability of being ≤ x

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(x, pdf, color='steelblue')
axes[0].set_title('Normal PDF  (μ=70, σ=10)')
axes[0].set_xlabel('x'); axes[0].grid(True, alpha=0.3)

axes[1].plot(x, cdf, color='tomato')
axes[1].set_title('Normal CDF')
axes[1].set_xlabel('x'); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

# Practical questions answered with the CDF
p_below_80 = sp_stats.norm.cdf(80, mu, sigma)
print(f'P(score < 80) = {p_below_80:.2%}')   # ~84%

p_between = sp_stats.norm.cdf(80, mu, sigma) - sp_stats.norm.cdf(60, mu, sigma)
print(f'P(60 < score < 80) = {p_between:.2%}')  # ~68% — the famous 68% rule

In [ ]:
# ── Other Common Distributions ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Binomial — count of successes in n trials (e.g. coin flips, pass/fail)
# n=20 trials, p=0.5 probability of success
k = np.arange(0, 21)
binom_pmf = sp_stats.binom.pmf(k, n=20, p=0.5)
axes[0].bar(k, binom_pmf, color='steelblue')
axes[0].set_title('Binomial (n=20, p=0.5)')
axes[0].set_xlabel('Number of successes')

# Poisson — count of events in a fixed interval (e.g. emails per hour)
# λ (lambda) = average number of events
k = np.arange(0, 20)
poisson_pmf = sp_stats.poisson.pmf(k, mu=5)  # average 5 events
axes[1].bar(k, poisson_pmf, color='mediumseagreen')
axes[1].set_title('Poisson (λ=5)')
axes[1].set_xlabel('Number of events')

# Uniform — every value equally likely (e.g. rolling a fair die)
x = np.linspace(-0.5, 1.5, 200)
uniform_pdf = sp_stats.uniform.pdf(x, loc=0, scale=1)  # uniform between 0 and 1
axes[2].plot(x, uniform_pdf, color='tomato', linewidth=2)
axes[2].set_title('Uniform (0, 1)')
axes[2].set_xlabel('x')

plt.tight_layout(); plt.show()

## 3. Sampling and the Central Limit Theorem

In [ ]:
# Central Limit Theorem (CLT):
# No matter the shape of the original distribution,
# the distribution of SAMPLE MEANS approaches a normal distribution
# as sample size grows. This is why the normal distribution is everywhere.

np.random.seed(42)
population = np.random.exponential(scale=2, size=100_000)  # skewed distribution

# Take many samples and record each sample's mean
sample_means_10  = [np.mean(np.random.choice(population, 10))  for _ in range(1000)]
sample_means_50  = [np.mean(np.random.choice(population, 50))  for _ in range(1000)]
sample_means_200 = [np.mean(np.random.choice(population, 200)) for _ in range(1000)]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, means, n in zip(axes, [sample_means_10, sample_means_50, sample_means_200], [10, 50, 200]):
    ax.hist(means, bins=30, color='steelblue', edgecolor='white')
    ax.set_title(f'Sample size = {n}')
    ax.set_xlabel('Sample mean')

plt.suptitle('Central Limit Theorem — sample means become normal', y=1.02)
plt.tight_layout(); plt.show()

## 4. Correlation and Covariance

In [ ]:
# Covariance — do two variables move together? (scale-dependent)
# Correlation — same idea but normalized to [-1, 1] (scale-independent)
#   +1 = perfect positive relationship
#    0 = no linear relationship
#   -1 = perfect negative relationship

np.random.seed(0)
x = np.random.randn(100)
y_pos = x + np.random.randn(100) * 0.5   # strong positive correlation
y_neg = -x + np.random.randn(100) * 0.5  # strong negative correlation
y_none = np.random.randn(100)             # no correlation

for label, y in [('Positive', y_pos), ('Negative', y_neg), ('None', y_none)]:
    r = np.corrcoef(x, y)[0, 1]  # corrcoef returns a 2x2 matrix; [0,1] is the cross-correlation
    print(f'{label:10s} correlation: r = {r:.3f}')

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, y, label in zip(axes, [y_pos, y_neg, y_none], ['Positive', 'Negative', 'None']):
    r = np.corrcoef(x, y)[0, 1]
    ax.scatter(x, y, alpha=0.5, color='steelblue')
    ax.set_title(f'{label}  (r={r:.2f})')
    ax.set_xlabel('x'); ax.set_ylabel('y')
    ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 5. Hypothesis Testing (basics)

In [ ]:
# Hypothesis testing answers: 'Is this difference real, or just random noise?'
#
# H0 (null hypothesis)     — no difference / no effect
# H1 (alternative)         — there IS a difference
# p-value                  — probability of seeing this result IF H0 were true
# If p < 0.05 → reject H0 (result is 'statistically significant')

np.random.seed(1)
group_a = np.random.normal(70, 10, 50)  # control group scores
group_b = np.random.normal(75, 10, 50)  # treatment group scores

# t-test — compares means of two independent groups
t_stat, p_value = sp_stats.ttest_ind(group_a, group_b)
print(f't-statistic : {t_stat:.3f}')
print(f'p-value     : {p_value:.4f}')
print('Significant!' if p_value < 0.05 else 'Not significant (could be chance)')

# Chi-square test — tests if two categorical variables are independent
# e.g. Is gender independent of product preference?
observed = np.array([[30, 20],   # [male: product A, product B]
                     [15, 35]])  # [female: product A, product B]
chi2, p, dof, expected = sp_stats.chi2_contingency(observed)
print(f'\nChi2 = {chi2:.3f}, p = {p:.4f}')
print('Variables are dependent!' if p < 0.05 else 'Variables appear independent')

## 6. Linear Algebra with NumPy

In [ ]:
# Vectors and matrices are the language of data science.
# numpy.linalg has all the essentials.

A = np.array([[2, 1],
              [5, 3]])
B = np.array([[1, 0],
              [2, 4]])

# Matrix operations
print('A + B:\n', A + B)              # element-wise addition
print('A @ B:\n', A @ B)              # matrix multiplication (dot product)
print('A.T:\n', A.T)                  # transpose — flip rows and columns

# Key properties
print('det(A)  :', np.linalg.det(A))  # determinant — 0 means matrix is singular (not invertible)
print('inv(A):\n', np.linalg.inv(A))  # inverse — A @ inv(A) = identity matrix
print('rank(A) :', np.linalg.matrix_rank(A))  # number of linearly independent rows/columns

In [ ]:
# Solving a system of linear equations
# 2x + y  = 5
# 5x + 3y = 13
# In matrix form: A @ x = b

A = np.array([[2, 1],
              [5, 3]])
b = np.array([5, 13])

x = np.linalg.solve(A, b)  # more numerically stable than computing inv(A) @ b
print('Solution:', x)       # [2. 1.] → x=2, y=1
print('Verify A @ x == b:', np.allclose(A @ x, b))  # True

In [ ]:
# Eigenvalues and Eigenvectors
# An eigenvector of A is a vector v where: A @ v = λ * v
# λ (eigenvalue) tells you how much the vector is scaled.
# Used in PCA, Google PageRank, physics simulations, and more.

A = np.array([[4, 2],
              [1, 3]])

eigenvalues, eigenvectors = np.linalg.eig(A)
print('Eigenvalues :', eigenvalues)    # [5. 2.]
print('Eigenvectors:\n', eigenvectors) # columns are the eigenvectors

# Verify: A @ v == λ * v for the first eigenvector
v0 = eigenvectors[:, 0]   # first eigenvector
lam0 = eigenvalues[0]     # corresponding eigenvalue
print('A @ v0      :', A @ v0)
print('λ0 * v0     :', lam0 * v0)
print('Equal?', np.allclose(A @ v0, lam0 * v0))  # True

In [ ]:
# Dot product and norms
# Dot product: measures similarity between two vectors
# Norm: the 'length' of a vector

u = np.array([1, 2, 3])
v = np.array([4, 5, 6])

print('Dot product :', np.dot(u, v))          # 1*4 + 2*5 + 3*6 = 32
print('||u|| (L2 norm):', np.linalg.norm(u))  # sqrt(1+4+9) ≈ 3.74

# Cosine similarity — angle between vectors, used in NLP and recommendations
cos_sim = np.dot(u, v) / (np.linalg.norm(u) * np.linalg.norm(v))
print('Cosine similarity:', round(cos_sim, 4))  # close to 1 → vectors point in same direction

## Exercise
1. Generate 1000 samples from a normal distribution (μ=50, σ=15). Compute mean, std, and the percentage of values within one standard deviation.
2. Run a t-test between two groups of your own data and interpret the p-value.
3. Solve the system: 3x + 2y = 12, x - y = 1 using `np.linalg.solve`.